# Notebook 2: Portfolio Summary (Easy - SQL + Python)
Combine SQL queries with Python pandas for portfolio breakdown and formatted output.

In [ ]:
%%sql -r loan_details
SELECT l.LOAN_ID, l.LOAN_TYPE, l.LOAN_AMOUNT, l.INTEREST_RATE, l.TERM_MONTHS,
       l.LOAN_STATUS, l.PURPOSE, l.MONTHLY_PAYMENT,
       c.FIRST_NAME || ' ' || c.LAST_NAME AS CUSTOMER_NAME,
       c.CREDIT_SCORE, c.ANNUAL_INCOME, c.STATE
FROM LOAN_DB.LENDING.LOANS l
JOIN LOAN_DB.LENDING.CUSTOMERS c ON l.CUSTOMER_ID = c.CUSTOMER_ID
ORDER BY l.LOAN_AMOUNT DESC

In [ ]:
import anthropic, json

client = anthropic.Anthropic()

sample_rows = loan_details.head(5).to_dict(orient='records')
num_rows = len(loan_details)
columns = list(loan_details.columns)

prompt = f"""
I ran a SQL query that joins LOANS and CUSTOMERS tables to fetch full loan details ordered by loan amount descending.

Result summary:
- Total rows returned: {num_rows}
- Columns: {columns}
- Top 5 rows (sample): {json.dumps(sample_rows, default=str, indent=2)}

Please:
1. Briefly confirm the query looks correct and the join is working as expected.
2. Comment on the spread of loan amounts and customer credit scores visible in the sample.
3. Flag any data quality concerns (nulls, unexpected values, etc.) based on the sample.
"""

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}]
)

print("=" * 60)
print("AI ANALYSIS: Loan Fetch Query")
print("=" * 60)
print(message.content[0].text)

In [ ]:
import pandas as pd

df = loan_details.copy()
print(f"Total Loans: {len(df)}")
print(f"Total Portfolio: ${df['LOAN_AMOUNT'].sum():,.2f}")
print(f"Average Loan: ${df['LOAN_AMOUNT'].mean():,.2f}")
print(f"\nLoan Types: {df['LOAN_TYPE'].nunique()}")
print(f"States Covered: {df['STATE'].nunique()}")

In [ ]:
import anthropic

client = anthropic.Anthropic()

total_loans = len(df)
total_portfolio = df['LOAN_AMOUNT'].sum()
avg_loan = df['LOAN_AMOUNT'].mean()
num_types = df['LOAN_TYPE'].nunique()
num_states = df['STATE'].nunique()

prompt = f"""
Here are the top-level stats for a loan portfolio:

- Total Loans: {total_loans}
- Total Portfolio Value: ${total_portfolio:,.2f}
- Average Loan Size: ${avg_loan:,.2f}
- Distinct Loan Types: {num_types}
- States Covered: {num_states}

Please:
1. Assess whether the average loan size seems reasonable relative to the portfolio size.
2. Comment on geographic and product diversification based on loan types and states.
3. Provide 1-2 key takeaways a portfolio manager should note from these headline numbers.
"""

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}]
)

print("=" * 60)
print("AI ANALYSIS: Portfolio Stats")
print("=" * 60)
print(message.content[0].text)

In [ ]:
summary = df.groupby('LOAN_TYPE').agg(
    count=('LOAN_ID', 'count'),
    total_amount=('LOAN_AMOUNT', 'sum'),
    avg_rate=('INTEREST_RATE', 'mean'),
    avg_credit_score=('CREDIT_SCORE', 'mean')
).round(2)

summary['pct_of_portfolio'] = (summary['total_amount'] / summary['total_amount'].sum() * 100).round(1)
summary = summary.sort_values('total_amount', ascending=False)
summary

In [ ]:
import anthropic

client = anthropic.Anthropic()

summary_dict = summary.reset_index().to_dict(orient='records')

prompt = f"""
Below is a portfolio breakdown grouped by loan type:

{summary_dict}

Each record has: LOAN_TYPE, count, total_amount, avg_rate, avg_credit_score, pct_of_portfolio.

Please:
1. Identify which loan type dominates the portfolio and whether this concentration is a risk.
2. Compare interest rates across loan types — are higher-risk types (lower credit scores) priced appropriately?
3. Highlight any loan type that stands out as either an opportunity or a concern.
"""

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}]
)

print("=" * 60)
print("AI ANALYSIS: Portfolio Breakdown by Loan Type")
print("=" * 60)
print(message.content[0].text)

In [ ]:
status_pivot = df.pivot_table(
    index='LOAN_TYPE',
    columns='LOAN_STATUS',
    values='LOAN_AMOUNT',
    aggfunc='sum',
    fill_value=0
)
status_pivot['TOTAL'] = status_pivot.sum(axis=1)
status_pivot = status_pivot.sort_values('TOTAL', ascending=False)
status_pivot

In [ ]:
import anthropic

client = anthropic.Anthropic()

pivot_dict = status_pivot.reset_index().to_dict(orient='records')
status_columns = [c for c in status_pivot.columns if c != 'TOTAL']

prompt = f"""
Below is a pivot table showing total loan amounts split by LOAN_TYPE (rows) and LOAN_STATUS (columns).
Status columns present: {status_columns}

Data:
{pivot_dict}

Please:
1. Identify which loan types have the highest proportion of non-active statuses (e.g. defaulted, late, charged-off).
2. Assess the overall portfolio health based on the status distribution.
3. Call out any loan type where the risk profile (bad statuses) is disproportionately high relative to its portfolio share.
"""

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}]
)

print("=" * 60)
print("AI ANALYSIS: Loan Amount by Type and Status")
print("=" * 60)
print(message.content[0].text)

In [ ]:
bins = [0, 10000, 50000, 100000, 250000, 700000]
labels = ['<10K', '10K-50K', '50K-100K', '100K-250K', '250K+']
df['SIZE_BUCKET'] = pd.cut(df['LOAN_AMOUNT'], bins=bins, labels=labels)

bucket_summary = df.groupby('SIZE_BUCKET', observed=True).agg(
    count=('LOAN_ID', 'count'),
    total=('LOAN_AMOUNT', 'sum'),
    avg_rate=('INTEREST_RATE', 'mean')
).round(2)
bucket_summary

In [ ]:
import anthropic

client = anthropic.Anthropic()

bucket_dict = bucket_summary.reset_index().to_dict(orient='records')

prompt = f"""
Below is a loan size distribution summary with buckets, loan counts, total amounts, and average interest rates:

{bucket_dict}

Buckets are: <10K, 10K-50K, 50K-100K, 100K-250K, 250K+

Please:
1. Describe the shape of the size distribution — is it skewed toward small or large loans?
2. Examine whether interest rates are appropriately tiered across loan sizes (larger loans typically carry lower rates due to lower relative origination cost).
3. Provide a recommendation on whether the lender should adjust its product mix or pricing strategy based on this distribution.
"""

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}]
)

print("=" * 60)
print("AI ANALYSIS: Loan Size Distribution")
print("=" * 60)
print(message.content[0].text)